<a href="https://colab.research.google.com/github/faiq191/244107020214-machine-learning/blob/main/JS11/JS11_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Step 0 - Unzip Dataset
```

In [26]:
import zipfile
import os
import shutil

zip_path = '/content/dataset.zip'

print("Mencoba mengekstrak dataset.zip yang diunggah...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')
print("Ekstraksi berhasil!")

# Periksa apakah ekstraksi menghasilkan folder bersarang (nested folder)
extracted_items = [item for item in os.listdir('/content') if os.path.isdir(os.path.join('/content', item)) and item not in ['.config', 'sample_data', '__MACOSX']]
print("Folder hasil ekstraksi:", extracted_items)

# Memastikan struktur folder 'dataset' berada di posisi yang benar
if len(extracted_items) == 1 and extracted_items[0] != 'dataset':
    nested_folder = os.path.join('/content', extracted_items[0])
    if os.path.exists('/content/dataset'):
        shutil.rmtree('/content/dataset')
    os.rename(nested_folder, '/content/dataset')

# Deteksi letak training_set & test_set
dataset_dir = '/content/dataset'
subdirs = os.listdir(dataset_dir) if os.path.exists(dataset_dir) else []
print("Isi di dalam folder dataset:", subdirs)

# Jika folder training_set / test_set berada di subfolder yang berbeda nama
for folder in subdirs:
    full_path = os.path.join(dataset_dir, folder)
    if os.path.isdir(full_path):
        if folder.lower() in ['train', 'training', 'train_set']:
            os.rename(full_path, os.path.join(dataset_dir, 'training_set'))
        elif folder.lower() in ['test', 'val', 'validation', 'test_set']:
            os.rename(full_path, os.path.join(dataset_dir, 'test_set'))

# Pastikan folder single_prediction siap
single_pred_dir = os.path.join(dataset_dir, 'single_prediction')
os.makedirs(single_pred_dir, exist_ok=True)

# Copy satu gambar sampel dari training set kucing jika file sampel tidak ada
try:
    train_cats_dir = os.path.join(dataset_dir, 'training_set', 'cats')
    if not os.listdir(single_pred_dir) and os.path.exists(train_cats_dir):
        sample_pic = os.listdir(train_cats_dir)[0]
        shutil.copy(os.path.join(train_cats_dir, sample_pic), os.path.join(single_pred_dir, 'cat_or_dog_1.jpg'))
        print("Berhasil membuat sampel gambar tunggal untuk prediksi di Step 5.")
except Exception as e:
    print("Catatan untuk Step 5:", e)

print("Struktur dataset akhir:", os.listdir(dataset_dir) if os.path.exists(dataset_dir) else "Belum siap")

Mencoba mengekstrak dataset.zip yang diunggah...
Ekstraksi berhasil!
Folder hasil ekstraksi: ['dataset']
Isi di dalam folder dataset: ['single_prediction', '.DS_Store', 'training_set', 'test_set']
Struktur dataset akhir: ['single_prediction', '.DS_Store', 'training_set', 'test_set']


```markdown
# Step 1 - Import Libraries
```

In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from tensorflow.keras.preprocessing import image

```markdown
# Step 2 - Data Preprocessing

In this stage we will perform a number of manipulations on the images used. The manipulations include pixel value normalization, shear correction, zoom augmentation, and flipping.
```

```markdown
## Step 2.1 - Training Data Preprocessing
```

In [28]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('/content/dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


```markdown
## Step 2.2 - Testing Data Preprocessing
```

In [29]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('/content/dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


```markdown
# Step 3 - Construction of the CNN Model
```

```markdown
## Step 3.1 - CNN Model Initialization
```

In [30]:
tf.keras.backend.clear_session()
cnn = tf.keras.models.Sequential()

```markdown
## Step 3.2 - Creation of Convolutional Layer 1
```

In [31]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


```markdown
## Step 3.3 - Creation of Pooling Layer 1
```

In [32]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

```markdown
## Step 3.4 - Creation of Convolutional Layer 2 and Pooling 2
```

In [33]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

```markdown
## Step 3.5 - Flattening
```

In [34]:
cnn.add(tf.keras.layers.Flatten())

```markdown
## Step 3.6 - Fully Connected Layer 1 (Input)
```

In [35]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

```markdown
## Step 3.7 - Fully Connected Layer 2 (Output)
```

In [36]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

```markdown
## Step 3.8 - Compile the CNN Model

Usage loss function: binary crossentropy because we are performing classification across only two classes, namely cat and dog.
```

In [37]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

```markdown
# Step 4 - Fit the CNN
```

In [38]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 68s 265ms/step - accuracy: 0.5876 - loss: 0.6756 - val_accuracy: 0.6520 - val_loss: 0.6394
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 261ms/step - accuracy: 0.6536 - loss: 0.6258 - val_accuracy: 0.6710 - val_loss: 0.5978
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 66s 264ms/step - accuracy: 0.6996 - loss: 0.5728 - val_accuracy: 0.7385 - val_loss: 0.5345
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 66s 262ms/step - accuracy: 0.7224 - loss: 0.5495 - val_accuracy: 0.7455 - val_loss: 0.5209
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 255ms/step - accuracy: 0.7451 - loss: 0.5155 - val_accuracy: 0.7520 - val_loss: 0.5141
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 261ms/step - accuracy: 0.7536 - loss: 0.4994 - val_accuracy: 0.7595 - val_loss: 0.5078
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 261ms/step - accuracy: 0.7634 - loss: 0.4794 - val_accuracy: 0.7690 - val_loss: 0.4833
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 278ms/step - accuracy: 0.7761 - loss: 0

```markdown
# Step 5 - Prediction with a Single Image

In this step, we will attempt to perform a prediction on a single dog or cat image.
```

In [39]:
test_image = image.load_img('/content/dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
# Normalisasi piksel gambar (rescale 1./255)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis = 0)

# Lakukan prediksi
result = cnn.predict(test_image)

print("Class Indices:", training_set.class_indices)
# Karena output layer menggunakan sigmoid, hasil di atas 0.5 adalah dog (kelas 1)
if result[0][0] > 0.5:
    prediction = 'dog'
else:
    prediction = 'cat'

print(f"Nilai output mentah: {result[0][0]}")
print(f"The prediction is: {prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Class Indices: {'cats': 0, 'dogs': 1}
Nilai output mentah: 0.9999189972877502
The prediction is: dog
